# 🧬 Module 1a-2: R Programming Fundamentals
### Logic, Decisions & Automated Loops
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

**Course:** Bioinformatics: From Bulk & scRNA-seq to Spatial Transcriptomics  
**Estimated Time:** 45–60 minutes

---

> 🤖 **How to Use AI in This Course:**  
> You are welcome to use AI as a **tutor**.  
> * **DO:** Ask AI to explain what a line does or why you got an error.  
> * **DO NOT:** Ask AI to write the answers for you. You learn by typing and solving challenges yourself!

---

### 🎯 What You Will Learn Today (In Simple Words):
1. **Compare Values:** How to ask R simple questions (like *"Does this spot have more than 500 reads?"*).
2. **Combine Rules:** How to use **AND** (`&`) and **OR** (`|`) to test two things at the same time.
3. **Make Decisions:** How to use `if` and `else` to keep good quality spots and drop bad ones.
4. **Filter Many Spots at Once:** How to use `ifelse()` on a whole list of cells in one step.
5. **Repeat Work (Loops):** How to run the same task on multiple sample files automatically.


---
## 1. Comparing Values (Relational Operators)

To filter biological data, we need to ask R questions that result in either `TRUE` or `FALSE`.

| Operator | Meaning | Example | Result |
| :--- | :--- | :--- | :--- |
| `==` | Is equal to? | `5 == 5` | `TRUE` |
| `!=` | Is NOT equal to? | `5 != 3` | `TRUE` |
| `>` | Greater than | `10 > 20` | `FALSE` |
| `<` | Less than | `10 < 20` | `TRUE` |
| `>=` | Greater than or equal to | `15 >= 15` | `TRUE` |
| `<=` | Less than or equal to | `12 <= 8` | `FALSE` |


In [ ]:
# [Sandbox] Asking R simple true/false questions
gene_count <- 1500

cat("Is gene_count exactly 1500? ", gene_count == 1500, "\n")
cat("Is gene_count not equal to 0? ", gene_count != 0, "\n")
cat("Is gene_count greater than 2000? ", gene_count > 2000, "\n")


In [ ]:
# [Biological Context] Cell sequencing depth threshold
# A common QC rule: A spot must have at least 500 UMIs to be considered real tissue
spot_umi_count <- 850
min_threshold <- 500

is_valid_spot <- spot_umi_count >= min_threshold

cat("Spot UMI count: ", spot_umi_count, "\n")
cat("Passes minimum threshold (>= 500)? ", is_valid_spot, "\n")


In [ ]:
# [Common Gotcha] Using = instead of ==
# In R:
# '=' assigns a value to a variable!
# '==' tests if two values are equal!

# BAD: This tries to overwrite x with 5
# if (x = 5) { ... }  # Syntax ERROR!

# GOOD:
x <- 10
cat("Is x equal to 5?", x == 5, "\n")


### ✍️ Exercise 2.1: Your Turn
**Task:**
1. Create a variable `measured_reads` with the value `1250`.
2. Test if `measured_reads` is strictly greater than `1000`. Store the logical result in `is_sufficient`.
3. Print `is_sufficient`.


In [ ]:
# --- Write your code for Exercise 2.1 below ---


<details>
<summary>💡 Click to view Solution for Exercise 2.1</summary>

```r
measured_reads <- 1250
is_sufficient <- measured_reads > 1000
cat("Is depth sufficient (> 1000)?", is_sufficient, "
")
```
</details>


---
## 2. Combining Rules (AND, OR, NOT)

Real biological quality control requires checking multiple conditions simultaneously.

* **AND (`&`)**: `TRUE` only if **both** conditions are true.
* **OR (`|`)**: `TRUE` if **at least one** condition is true.
* **NOT (`!`)**: Inverts truth (turns `TRUE` to `FALSE`, and `FALSE` to `TRUE`).


In [ ]:
# [Sandbox] Combining conditions
has_good_depth <- TRUE
has_low_contamination <- TRUE

# AND (&): Both must be TRUE
cat("Passes both tests (AND):", has_good_depth & has_low_contamination, "\n")

# NOT (!): Inverts the value
cat("Inverted test (!has_good_depth):", !has_good_depth, "\n")


In [ ]:
# [Biological Context] Dual-Gate Spatial Quality Control
# Real-world rule: Keep spots with > 1000 detected genes AND < 15% mitochondrial reads
detected_genes <- 1450
mito_fraction <- 7.2  # 7.2%

# Both criteria must pass:
passes_quality_gate <- (detected_genes > 1000) & (mito_fraction < 15.0)

cat("Detected Genes:  ", detected_genes, "(Threshold: > 1000)\n")
cat("Mito Fraction:   ", mito_fraction, "% (Threshold: < 15%)\n")
cat("Passes Dual Gate:", passes_quality_gate, "\n")


In [ ]:
# [Common Gotcha] Single '&' vs Double '&&'
# In R:
# '&' is element-wise: It tests every single element in a list/vector.
# '&&' only tests the VERY FIRST element and ignores the rest!
#
# For data filtering across cells, ALWAYS use single '&' and single '|'!
vec1 <- c(TRUE, FALSE)
vec2 <- c(TRUE, TRUE)

cat("Single & (Element-wise):\n")
print(vec1 & vec2)  # Returns: TRUE FALSE


### ✍️ Exercise 2.2: Your Turn
**Task:**
A spatial sequencing spot has:
* `total_umis <- 4500`
* `ribosomal_pct <- 22.5`

Write code to test if this spot is a **High Quality Cell**:
1. It must have `total_umis >= 2000`.
2. It must have `ribosomal_pct <= 30.0`.
3. Save the result in `is_high_quality` and print it.


In [ ]:
# --- Write your code for Exercise 2.2 below ---


<details>
<summary>💡 Click to view Solution for Exercise 2.2</summary>

```r
total_umis <- 4500
ribosomal_pct <- 22.5

is_high_quality <- (total_umis >= 2000) & (ribosomal_pct <= 30.0)
cat("High quality spot:", is_high_quality, "
")
```
</details>


---
## 3. Making Decisions (`if`, `else if`, `else`)

Conditional statements let R choose different paths of code based on the data.

```
if (condition) {
  # Code runs if condition is TRUE
} else if (another_condition) {
  # Code runs if the second condition is TRUE
} else {
  # Code runs if none of the above are TRUE
}
```


In [ ]:
# [Sandbox] Simple 3-tier decision logic
test_score <- 85

if (test_score >= 90) {
  result_tier <- "Tier 1: Excellent"
} else if (test_score >= 70) {
  result_tier <- "Tier 2: Good"
} else {
  result_tier <- "Tier 3: Needs Review"
}

cat("Score:", test_score, "->", result_tier, "\n")


In [ ]:
# [Biological Context] Cell Viability Classifier
# In single-cell and spatial RNA-seq, high mitochondrial fraction indicates dying/lysed cells.
cell_mito_pct <- 18.5

if (cell_mito_pct > 20.0) {
  status <- "DROP: Severe cell membrane rupture"
} else if (cell_mito_pct > 10.0) {
  status <- "FLAG: Moderate stress or high metabolic activity"
} else {
  status <- "PASS: Healthy, intact cell"
}

cat("Mitochondrial Content:", cell_mito_pct, "%\n")
cat("QC Action Plan:       ", status, "\n")


In [ ]:
# [Common Gotcha] Placing 'else' on a brand new line
# In R scripts, if 'else' is on a new line after the closing brace,
# R thinks the 'if' block finished and throws a syntax error!

# BAD:
# if (x > 5) {
#   print("high")
# }
# else {   # ERROR: unexpected 'else'
#   print("low")
# }

# GOOD: Always place '} else {' on the SAME line!
val <- 4
if (val > 5) {
  print("High")
} else {
  print("Low")
}


### ✍️ Exercise 2.3: Your Turn
**Task:**
Write an `if` / `else if` / `else` decision statement for sequencing depth saturation:
* Create `saturation <- 65` (percentage).
* If `saturation >= 80`, set `decision <- "High saturation (sequenced to completion)"`.
* Else if `saturation >= 50`, set `decision <- "Moderate saturation (sufficient depth)"`.
* Else, set `decision <- "Low saturation (more sequencing recommended)"`.
* Print `decision`.


In [ ]:
# --- Write your code for Exercise 2.3 below ---


<details>
<summary>💡 Click to view Solution for Exercise 2.3</summary>

```r
saturation <- 65

if (saturation >= 80) {
  decision <- "High saturation (sequenced to completion)"
} else if (saturation >= 50) {
  decision <- "Moderate saturation (sufficient depth)"
} else {
  decision <- "Low saturation (more sequencing recommended)"
}

cat("Sequencing Decision:", decision, "
")
```
</details>


---
## 4. Vectorized Filtering with `ifelse()`

A standard `if` statement can only test **one single value** at a time.
In bioinformatics, we have thousands of spots in a single vector!

To test an entire list/vector of numbers at once, R provides the magical `ifelse()` function:

```
ifelse(test_vector, value_if_true, value_if_false)
```


In [ ]:
# [Sandbox] Testing a whole vector in one line
scores <- c(45, 88, 92, 60, 78)

# Label every score above 70 as "Pass", otherwise "Review"
labels <- ifelse(scores >= 70, "Pass", "Review")

# Combine into a quick view
print(data.frame(Score = scores, Status = labels))


In [ ]:
# [Biological Context] Classifying Spatial Tissue Spots
# Imagine we have 6 capture spots on a Visium slide with different UMI counts:
spot_umis <- c(350, 1820, 4100, 120, 2450, 480)

# If UMI count >= 500, it is covered by Tissue; otherwise Background
spot_types <- ifelse(spot_umis >= 500, "Tissue", "Background")

# Inspect the result
spot_summary <- data.frame(
  Spot_Index = 1:6,
  Raw_UMIs = spot_umis,
  Tissue_Status = spot_types
)
print(spot_summary)


In [ ]:
# [Common Gotcha] Using standard 'if' on a vector
# If you try to pass a vector to standard 'if', R will warn you:
# "the condition has length > 1 and only the first element will be used"

numbers <- c(10, 2, 8)
# if (numbers > 5) { print("Yes") } # Generates WARNING!

# Rule:
# Use 'if' for a single scalar value.
# Use 'ifelse()' for vectors / lists of numbers!


### ✍️ Exercise 2.4: Your Turn
**Task:**
You have a vector of 5 gene expression counts:
```r
gene_counts <- c(0, 14, 0, 85, 3)
```
1. Use `ifelse()` to create a vector named `expression_status`:
   * If count is `> 0`, label it `"Expressed"`.
   * Otherwise, label it `"Silent"`.
2. Print `expression_status`.


In [ ]:
# --- Write your code for Exercise 2.4 below ---
gene_counts <- c(0, 14, 0, 85, 3)


<details>
<summary>💡 Click to view Solution for Exercise 2.4</summary>

```r
gene_counts <- c(0, 14, 0, 85, 3)
expression_status <- ifelse(gene_counts > 0, "Expressed", "Silent")
print(expression_status)
```
</details>


---
## 5. Repeating Tasks Automatically (Loops)

A **`for` loop** tells R to repeat a block of code for every item in a list.

```
for (item in collection) {
  # Do something with item
}
```


In [ ]:
# [Sandbox] Basic for loop over numbers
for (counter in 1:4) {
  cat("Running iteration number:", counter, "\n")
}


In [ ]:
# [Biological Context] Processing multiple sample files
# Suppose our spatial study has 3 patient tissue sections:
patient_samples <- c("Patient_01_SectionA", "Patient_02_SectionB", "Patient_03_SectionC")

for (sample in patient_samples) {
  file_to_load <- paste0("data/raw/", sample, "_counts.csv")
  cat("Loading sample:", sample, "from ->", file_to_load, "\n")
}


In [ ]:
# [Common Gotcha] Growing vectors inside loops
# Beginners often do this:
# results <- c()
# for (i in 1:1000) { results <- c(results, i^2) }
#
# Why this is slow: R has to copy the entire list in memory on EVERY iteration!
# In Week 2, we will learn faster vectorized alternatives ('apply' and pre-allocation).


### ✍️ Exercise 2.5: Your Turn
**Task:**
You have a vector of 4 marker genes:
```r
marker_genes <- c("CD8A", "CD4", "MS4A1", "EPCAM")
```
1. Write a `for` loop that loops through each gene.
2. Inside the loop, print the message: `"Analyzing marker gene: [GENE]"` using `cat()` or `paste()`.


In [ ]:
# --- Write your code for Exercise 2.5 below ---
marker_genes <- c("CD8A", "CD4", "MS4A1", "EPCAM")


<details>
<summary>💡 Click to view Solution for Exercise 2.5</summary>

```r
marker_genes <- c("CD8A", "CD4", "MS4A1", "EPCAM")

for (gene in marker_genes) {
  cat("Analyzing marker gene:", gene, "
")
}
```
</details>


---
## 🏁 Week 1 Complete! Summary & Looking Ahead

Congratulations! You have finished both fundamentals notebooks for Week 1. You now know:
* How to ask R comparison questions using `==`, `!=`, `>`, `<`, `>=`, and `<=`.
* How to combine rules with AND (`&`) and OR (`|`).
* How to use `if` / `else` to classify biological data.
* How to filter entire vectors of cells simultaneously with `ifelse()`.
* How to run repetitive tasks across samples with `for` loops.

---
### 📅 Coming Up in Week 2:
In Week 2, we will move from single variables into R's powerful biological data containers:
* **`01b1_Vectors_and_Lists.ipynb`**: Master 1-based indexing, nested lists, and regular expressions for gene patterns.
* **`01b2_Matrices_and_DataFrames.ipynb`**: Build 2D gene expression matrices, work with factors, clean missing data (`NA`), and load data at high speed!
